# Model Kaydetme, Yeniden Yükleme ve Canlı Çıkarım (Inference Pipeline)

Bu modül; eğitilen bir yapay sinir ağının üretime (production) hazır hale getirilmesi için modern **Keras v3 `.keras` formatında serileştirilmesi**, diskten yüklenmesi ve dışarıdan gelen ham veriler üzerinde uçtan uca çıkarım (inference) yapan bir tahmin fonksiyonunun tasarlanmasını inceler.

---

## 1. Keras Model Kaydetme Formatları

- **`.keras` (Zip Tabanlı Yerel Format - Önerilen):** Keras v3'ün modern standardıdır. Model mimarisini, katman yapılandırmasını, ağırlıkları ve optimizer durumunu tek bir hafif zip arşivinde saklar.
- **`.h5` (HDF5 Legacy Format):** Eski Keras sürümlerinin standardıdır; ancak özel katmanlar ve Keras 3 uyumluluğunda kısıtları vardır.
- **`SavedModel` (TensorFlow Dizin Formatı):** TensorFlow Serving ve C++ dağıtımları için kullanılan protobuf tabanlı dizin yapısıdır.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow Sürümü: {tf.__version__}")

# 1. Hızlı Bir MNIST Rakam Sınıflandırıcı Eğitme
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalizasyon ve Boyut Genişletme
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)

model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 2 epoch hızlı eğitim
model.fit(X_train[:10000], y_train[:10000], epochs=2, batch_size=64, validation_split=0.1)


## 2. Modelin Diske Kaydedilmesi ve Doğrulanması

In [ ]:
# Modern .keras formatında kaydetme
model_filename = 'mnist_classifier.keras'
model.save(model_filename)
print(f"Model başarıyla '{model_filename}' olarak kaydedildi.")

# Sadece ağırlıkları (weights) kaydetme alternatifi
model.save_weights('mnist_weights.weights.h5')
print("Model ağırlıkları 'mnist_weights.weights.h5' olarak kaydedildi.")


## 3. Bağımsız Bir Oturumda Modelin Yüklenmesi

In [ ]:
# Eğitilen modeli diskten yükleme
loaded_model = keras.models.load_model('mnist_classifier.keras')
print("Model diskten başarıyla yüklendi!")

# Test kümesi üzerinde doğrulama
test_loss, test_acc = loaded_model.evaluate(X_test[:500], y_test[:500], verbose=0)
print(f"Yüklenen Model Test Doğruluğu: %{test_acc * 100:.2f}")


## 4. Canlı Çıkarım (Production Inference Pipeline) Fonksiyonu

Üretim ortamında bir model doğrudan ham girdi alamaz; önce boyut, ölçek ve batch uyumluluğundan geçirilmelidir.


In [ ]:
def predict_digit(raw_image_28x28, model):
    """
    Ham bir 28x28 gri seviye görseli alır, ön işler ve en olası sınıfı döndürür.
    """
    # 1. Tip ve değer aralığı kontrolü
    img = np.array(raw_image_28x28, dtype=np.float32)
    if img.max() > 1.0:
        img = img / 255.0  # [0, 255] -> [0, 1]
        
    # 2. Boyut uyumluluğu: (28, 28) -> (1, 28, 28, 1)
    if img.ndim == 2:
        img = np.expand_dims(img, axis=(0, -1))
    elif img.ndim == 3:
        img = np.expand_dims(img, axis=0)
        
    # 3. Model Tahmini
    probabilities = model.predict(img, verbose=0)[0]
    predicted_class = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_class])
    
    return {
        "tahmin_edilen_rakam": predicted_class,
        "guven_skoru": f"%{confidence * 100:.2f}",
        "tum_olasiliklar": {int(i): round(float(p), 4) for i, p in enumerate(probabilities)}
    }

# Rastgele bir test görseli üzerinde deneme
sample_idx = 42
sample_img = X_test[sample_idx, :, :, 0]
actual_label = y_test[sample_idx]

result = predict_digit(sample_img, loaded_model)
print(f"Gerçek Etiket        : {actual_label}")
print(f"Modelin Tahmini      : {result['tahmin_edilen_rakam']}")
print(f"Güven Skoru (Softmax): {result['guven_skoru']}")
